# 03 — Grid Search (refatorado)

Versão limpa do notebook didático original (`03_grid_search_original.ipynb`).

Toda a lógica vive em `src/`. Este notebook é apenas o **consumidor** dos módulos — boa prática para manter código testável, reusável pela API e pelo Streamlit, e sem lógica duplicada.

## Pipeline
1. Carrega os dois CSVs já pré-processados pelas Partes 1 e 2 da série.
2. Reserva 20% como **holdout** (não entra no grid_search — métrica honesta).
3. Roda `GridSearchCV(KFold=5)` para RandomForest e XGBoost no log-target.
4. Avalia ambos no holdout (RMSLE + MAE em USD + R²).
5. Salva o vencedor em `models/best_model.pkl` + `models/metadata.json`.

## Diferenças vs. o original
- Sem `max_features='auto'` (deprecated em scikit-learn ≥ 1.3).
- Target em `np.log1p` para alinhar com a métrica oficial Kaggle (RMSLE).
- Holdout reservado fora do grid (`split_holdout`).
- Persistência via `joblib.dump` + `metadata.json`.
- Grid enxuto (12 + 24 combos vs. ~750 fits do original).

In [1]:
import sys
from pathlib import Path

# Permite executar o notebook a partir de notebooks/ sem instalar o pacote
ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data import load_data, split_holdout
from src.preprocessing import prepare_features
from src.train import run_grid_search, evaluate_holdout, save_model

## 1. Carregar dados e separar holdout

`load_data()` retorna `(treino, teste)` direto dos CSVs pré-processados em `data/`.
O teste (`teste`) só vai ser usado para gerar a submissão Kaggle (não há `SalePrice`).

`split_holdout` separa 20% do treino para a métrica final.

In [2]:
treino, teste = load_data()
print('treino:', treino.shape, '| teste:', teste.shape)

treino_dev, treino_holdout = split_holdout(treino)
print('dev:', treino_dev.shape, '| holdout:', treino_holdout.shape)

treino: (1460, 85) | teste: (1459, 84)
dev: (1168, 85) | holdout: (292, 85)


## 2. Preparar features

`prepare_features` seleciona as 48 colunas numéricas (descartando `Id` e `SalePrice`)
e aplica `np.log1p` ao target.

In [3]:
X_dev, y_dev = prepare_features(treino_dev)
X_h, y_h = prepare_features(treino_holdout)
print('X_dev:', X_dev.shape, '| X_h:', X_h.shape)
print('y_dev range (log):', round(float(y_dev.min()), 2), '..', round(float(y_dev.max()), 2))

X_dev: (1168, 48) | X_h: (292, 48)
y_dev range (log): 10.46 .. 13.52


## 3. Grid search — RandomForest

12 combos × 5 folds = 60 fits.
Scoring: `neg_root_mean_squared_error` no log-target (= RMSLE).

In [4]:
best_rf, summary_rf = run_grid_search('rf', X_dev, y_dev)
print('RF best params:', summary_rf['best_params'])
print('RF CV RMSLE:', round(summary_rf['best_score_rmsle_cv'], 4))

RF best params: {'max_depth': 20, 'max_features': 0.5, 'n_estimators': 500}
RF CV RMSLE: 0.1403


## 4. Grid search — XGBoost

24 combos × 5 folds = 120 fits.

In [5]:
best_xgb, summary_xgb = run_grid_search('xgb', X_dev, y_dev)
print('XGB best params:', summary_xgb['best_params'])
print('XGB CV RMSLE:', round(summary_xgb['best_score_rmsle_cv'], 4))

XGB best params: {'colsample_bytree': 0.7, 'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 300}
XGB CV RMSLE: 0.1314


## 5. Avaliação no holdout

RMSLE / MAE em USD / R² para cada modelo.

In [6]:
metrics_rf = evaluate_holdout(best_rf, X_h, y_h)
metrics_xgb = evaluate_holdout(best_xgb, X_h, y_h)

import pandas as pd
pd.DataFrame({'rf': metrics_rf, 'xgb': metrics_xgb}).T

,rmsle,mae_usd,r2
rf,0.145239,17306.933129,0.883058
xgb,0.139103,17194.912765,0.883436


## 6. Salvar vencedor

Critério de seleção: **menor RMSLE no CV** (mesma decisão do `src/train.py::main`).
Persiste em `models/best_model.pkl` + `models/metadata.json`.

In [7]:
rf_cv = summary_rf['best_score_rmsle_cv']
xgb_cv = summary_xgb['best_score_rmsle_cv']

if rf_cv < xgb_cv:
    winner, name, winner_metrics = best_rf, 'rf', metrics_rf
else:
    winner, name, winner_metrics = best_xgb, 'xgb', metrics_xgb

metadata = {
    'algorithm': name,
    'hyperparams': winner.get_params(),
    'metrics': winner_metrics,
    'comparison': {
        'rf': {**metrics_rf, 'cv_rmsle': rf_cv, 'best_params': summary_rf['best_params']},
        'xgb': {**metrics_xgb, 'cv_rmsle': xgb_cv, 'best_params': summary_xgb['best_params']},
    },
}
save_model(winner, metadata)
print(f'Vencedor: {name}. Modelo salvo em models/best_model.pkl')

Vencedor: xgb. Modelo salvo em models/best_model.pkl
